# Feature Engineering & Feature Importance Analysis

This notebook builds all price-based technical features, cross-ticker features, and calendar features from the raw OHLCV data. It then runs feature importance analysis to identify and select the most significant features for the modeling phase.

**Inputs:** `price.csv` (7 tickers, 241 trading days each)

**Outputs:** `features_price.csv` -- engineered feature matrix with importance rankings

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import ta
from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import IsolationForest
import xgboost as xgb
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error, mean_absolute_error
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['figure.dpi'] = 100

DATA_DIR = '.'

## 1. Load and Prepare Data

In [ ]:
price_df = pd.read_csv(f'{DATA_DIR}/price.csv', parse_dates=['date'])
price_df = price_df.sort_values(['ticker', 'date']).reset_index(drop=True)

tickers = sorted(price_df['ticker'].unique())
print(f'Tickers: {tickers}')
print(f'Shape: {price_df.shape}')
print(f'Date range: {price_df["date"].min().date()} to {price_df["date"].max().date()}')

## 2. Outlier Detection

Apply Isolation Forest per ticker to flag extreme anomalies, plus quantile clipping on returns (from SOTA Notebooks 3 & 4).

In [ ]:
price_df['return_1d'] = price_df.groupby('ticker')['close'].pct_change()

iso_forest = IsolationForest(contamination=0.01, random_state=42)
outlier_counts = {}

for ticker in tickers:
    mask = price_df['ticker'] == ticker
    subset = price_df.loc[mask, ['open', 'high', 'low', 'close', 'volume']].dropna()
    labels = iso_forest.fit_predict(subset)
    n_outliers = (labels == -1).sum()
    outlier_counts[ticker] = n_outliers
    price_df.loc[subset.index[labels == -1], 'is_outlier'] = True

price_df['is_outlier'] = price_df['is_outlier'].fillna(False)

print('=== Outliers detected per ticker (Isolation Forest, contamination=0.01) ===')
for t, c in outlier_counts.items():
    print(f'  {t}: {c} outliers')
print(f'\nTotal outliers: {price_df["is_outlier"].sum()} / {len(price_df)}')
print('Note: Outliers are flagged but NOT removed -- the model can learn from them.')

## 3. Technical Indicator Features

All features are computed **per ticker** to avoid cross-ticker contamination.

In [ ]:
def compute_technical_features(df):
    """Compute all technical indicators for a single ticker's dataframe."""
    out = df.copy()
    close = out['close']
    high = out['high']
    low = out['low']
    volume = out['volume']

    # --- Returns ---
    out['return_1d'] = close.pct_change(1)
    out['return_5d'] = close.pct_change(5)
    out['return_10d'] = close.pct_change(10)
    out['return_20d'] = close.pct_change(20)
    out['log_return_1d'] = np.log(close / close.shift(1))

    # --- Moving Averages ---
    out['sma_5'] = ta.trend.sma_indicator(close, window=5)
    out['sma_10'] = ta.trend.sma_indicator(close, window=10)
    out['sma_20'] = ta.trend.sma_indicator(close, window=20)
    out['ema_12'] = ta.trend.ema_indicator(close, window=12)
    out['ema_26'] = ta.trend.ema_indicator(close, window=26)

    # Price relative to MAs (normalized distance)
    out['close_to_sma5'] = (close - out['sma_5']) / out['sma_5']
    out['close_to_sma20'] = (close - out['sma_20']) / out['sma_20']
    out['sma5_to_sma20'] = (out['sma_5'] - out['sma_20']) / out['sma_20']

    # --- Momentum ---
    out['rsi_14'] = ta.momentum.rsi(close, window=14)
    macd_obj = ta.trend.MACD(close, window_slow=26, window_fast=12, window_sign=9)
    out['macd'] = macd_obj.macd()
    out['macd_signal'] = macd_obj.macd_signal()
    out['macd_hist'] = macd_obj.macd_diff()
    stoch = ta.momentum.StochasticOscillator(high, low, close, window=14, smooth_window=3)
    out['stoch_k'] = stoch.stoch()
    out['stoch_d'] = stoch.stoch_signal()

    # --- Volatility ---
    bb = ta.volatility.BollingerBands(close, window=20, window_dev=2)
    out['bb_width'] = bb.bollinger_wband()
    out['bb_pctb'] = bb.bollinger_pband()
    out['atr_14'] = ta.volatility.average_true_range(high, low, close, window=14)
    out['volatility_5d'] = out['return_1d'].rolling(5).std()
    out['volatility_10d'] = out['return_1d'].rolling(10).std()
    out['volatility_20d'] = out['return_1d'].rolling(20).std()

    # --- Price Range (from SOTA Notebook 4) ---
    out['price_range'] = high - low
    out['price_range_pct'] = (high - low) / close

    # --- Volume features ---
    out['obv'] = ta.volume.on_balance_volume(close, volume)
    out['volume_sma_20'] = volume.rolling(20).mean()
    out['volume_ratio'] = volume / out['volume_sma_20']
    out['volume_change'] = volume.pct_change()

    # --- Lag features (for tree-based models) ---
    for lag in range(1, 6):
        out[f'close_lag_{lag}'] = close.shift(lag)
        out[f'return_lag_{lag}'] = out['return_1d'].shift(lag)

    # --- Calendar features ---
    out['day_of_week'] = out['date'].dt.dayofweek
    out['month'] = out['date'].dt.month
    out['is_month_start'] = out['date'].dt.is_month_start.astype(int)
    out['is_month_end'] = out['date'].dt.is_month_end.astype(int)

    return out

In [ ]:
feature_dfs = []
for ticker in tickers:
    sub = price_df[price_df['ticker'] == ticker].copy()
    sub = compute_technical_features(sub)
    feature_dfs.append(sub)

features_df = pd.concat(feature_dfs, ignore_index=True)
print(f'Feature matrix shape (before NaN drop): {features_df.shape}')
print(f'Columns: {features_df.shape[1]}')
print(f'\nNaN counts per feature (top 15):')
display(features_df.isnull().sum().sort_values(ascending=False).head(15))

## 4. Cross-Ticker (Market) Features

In [ ]:
market_return = (
    features_df
    .groupby('date')['return_1d']
    .mean()
    .rename('market_return')
)

features_df = features_df.merge(market_return, on='date', how='left')
features_df['relative_return'] = features_df['return_1d'] - features_df['market_return']

market_vol = (
    features_df
    .groupby('date')['return_1d']
    .std()
    .rename('market_volatility')
)
features_df = features_df.merge(market_vol, on='date', how='left')

print(f'Added cross-ticker features: market_return, relative_return, market_volatility')
print(f'Feature matrix shape: {features_df.shape}')

## 5. Target Variable

In [ ]:
features_df['target_next_close'] = features_df.groupby('ticker')['close'].shift(-1)
features_df['target_next_return'] = features_df.groupby('ticker')['return_1d'].shift(-1)
features_df['target_direction'] = (features_df['target_next_return'] > 0).astype(int)

print('Target variables created:')
print('  - target_next_close:  next day closing price')
print('  - target_next_return: next day return (for alternative formulation)')
print('  - target_direction:   1 if next day up, 0 if down')
print(f'\nDirection distribution:')
display(features_df['target_direction'].value_counts(normalize=True))

## 6. Handle NaN Rows

Technical indicators produce NaNs for the first N rows (warmup period). We drop these rows and the last row per ticker (no target available).

In [ ]:
rows_before = len(features_df)

exclude_cols = ['date', 'ticker', 'is_outlier']
feature_cols_all = [
    c for c in features_df.columns
    if c not in exclude_cols
    and not c.startswith('target_')
    and c not in ['open', 'high', 'low', 'close', 'volume']
]

features_clean = features_df.dropna(subset=feature_cols_all + ['target_next_close']).copy()

rows_after = len(features_clean)
print(f'Rows before NaN drop: {rows_before}')
print(f'Rows after NaN drop:  {rows_after} (lost {rows_before - rows_after} warmup/tail rows)')
print(f'Rows per ticker: ~{rows_after // len(tickers)}')
print(f'\nFeature columns: {len(feature_cols_all)}')
print(feature_cols_all)

## 7. Feature Correlation Analysis

In [ ]:
corr_matrix = features_clean[feature_cols_all].corr()

fig, ax = plt.subplots(figsize=(20, 16))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, cmap='RdYlBu_r', center=0,
            vmin=-1, vmax=1, square=True, ax=ax,
            xticklabels=True, yticklabels=True,
            cbar_kws={'shrink': 0.8})
ax.set_title('Feature Correlation Matrix', fontsize=16, fontweight='bold')
ax.tick_params(axis='both', labelsize=7)
plt.tight_layout()
plt.show()

In [ ]:
# Identify highly correlated feature pairs (|r| > 0.90)
high_corr_threshold = 0.90
upper = corr_matrix.where(mask == False)
high_corr_pairs = []

for col in upper.columns:
    for idx in upper.index:
        val = upper.loc[idx, col]
        if pd.notna(val) and abs(val) > high_corr_threshold and idx != col:
            high_corr_pairs.append((idx, col, round(val, 3)))

high_corr_pairs.sort(key=lambda x: abs(x[2]), reverse=True)

print(f'=== Highly correlated pairs (|r| > {high_corr_threshold}) ===')
print(f'Found {len(high_corr_pairs)} pairs:\n')
for f1, f2, r in high_corr_pairs[:20]:
    print(f'  {f1:<25} vs {f2:<25}  r = {r:+.3f}')

## 8. Feature Importance -- XGBoost

Train a quick XGBoost model per ticker to rank features by importance. Uses time-based split to avoid leakage.

In [ ]:
importance_results = {}

for ticker in tickers:
    sub = features_clean[features_clean['ticker'] == ticker].sort_values('date')
    X = sub[feature_cols_all].values
    y = sub['target_next_close'].values

    split_idx = int(len(X) * 0.8)
    X_train, X_test = X[:split_idx], X[split_idx:]
    y_train, y_test = y[:split_idx], y[split_idx:]

    model = xgb.XGBRegressor(
        n_estimators=200,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        verbosity=0
    )
    model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)

    y_pred = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)

    importance = dict(zip(feature_cols_all, model.feature_importances_))
    importance_results[ticker] = importance

    print(f'{ticker}: RMSE={rmse:.2f}, MAE={mae:.2f}')

importance_df = pd.DataFrame(importance_results)
importance_df['mean_importance'] = importance_df.mean(axis=1)
importance_df = importance_df.sort_values('mean_importance', ascending=False)

print(f'\n=== Top 20 Features by Mean Importance Across Tickers ===')
display(importance_df['mean_importance'].head(20))

In [ ]:
# Feature importance visualization
top_n = 25
top_features = importance_df.head(top_n)

fig, axes = plt.subplots(1, 2, figsize=(20, 10))

# Mean importance bar chart
top_features['mean_importance'].plot(
    kind='barh', ax=axes[0], color='steelblue', edgecolor='white'
)
axes[0].set_xlabel('Mean Importance (gain)', fontsize=12)
axes[0].set_title(f'Top {top_n} Features by Mean Importance', fontsize=14, fontweight='bold')
axes[0].invert_yaxis()

# Per-ticker heatmap of importance
sns.heatmap(
    top_features[tickers],
    cmap='YlOrRd', annot=True, fmt='.3f',
    ax=axes[1], cbar_kws={'shrink': 0.8}
)
axes[1].set_title(f'Top {top_n} Feature Importance Per Ticker', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Ticker')

plt.tight_layout()
plt.show()

## 9. Feature Importance -- Correlation with Target

In [ ]:
target_corr_results = {}

for ticker in tickers:
    sub = features_clean[features_clean['ticker'] == ticker]
    corrs = sub[feature_cols_all].corrwith(sub['target_next_return']).abs()
    target_corr_results[ticker] = corrs

target_corr_df = pd.DataFrame(target_corr_results)
target_corr_df['mean_abs_corr'] = target_corr_df.mean(axis=1)
target_corr_df = target_corr_df.sort_values('mean_abs_corr', ascending=False)

print('=== Top 20 Features by Mean |correlation| with Next-Day Return ===')
display(target_corr_df['mean_abs_corr'].head(20))

fig, ax = plt.subplots(figsize=(12, 8))
target_corr_df['mean_abs_corr'].head(25).plot(
    kind='barh', ax=ax, color='coral', edgecolor='white'
)
ax.set_xlabel('Mean |Correlation| with Next-Day Return', fontsize=12)
ax.set_title('Features Most Correlated with Target', fontsize=14, fontweight='bold')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## 10. Feature Selection

Combine XGBoost importance + target correlation + redundancy removal to select the final feature set.

In [ ]:
# Combine both rankings into a unified score
importance_rank = importance_df['mean_importance'].rank(ascending=False)
corr_rank = target_corr_df['mean_abs_corr'].rank(ascending=False)

combined = pd.DataFrame({
    'xgb_importance': importance_df['mean_importance'],
    'xgb_rank': importance_rank,
    'target_corr': target_corr_df['mean_abs_corr'],
    'corr_rank': corr_rank,
}).dropna()

combined['combined_rank'] = (combined['xgb_rank'] + combined['corr_rank']) / 2
combined = combined.sort_values('combined_rank')

print('=== Combined Feature Ranking (lower = better) ===')
display(combined.head(30))

In [ ]:
def remove_redundant_features(feature_list, corr_matrix, threshold=0.95):
    """Remove features with correlation > threshold, keeping the higher-ranked one."""
    selected = list(feature_list)
    removed = []

    for i in range(len(selected)):
        if selected[i] is None:
            continue
        for j in range(i + 1, len(selected)):
            if selected[j] is None:
                continue
            if abs(corr_matrix.loc[selected[i], selected[j]]) > threshold:
                removed.append((selected[j], selected[i], corr_matrix.loc[selected[i], selected[j]]))
                selected[j] = None

    selected = [f for f in selected if f is not None]
    return selected, removed


ranked_features = combined.index.tolist()
corr_for_selection = features_clean[feature_cols_all].corr()

selected_features, removed_features = remove_redundant_features(
    ranked_features, corr_for_selection, threshold=0.95
)

print(f'=== Feature Selection Results ===')
print(f'Total features before:  {len(ranked_features)}')
print(f'Removed (redundant):    {len(removed_features)}')
print(f'Selected features:      {len(selected_features)}')

if removed_features:
    print(f'\nRemoved features (corr > 0.95 with a higher-ranked feature):')
    for dropped, kept, r in removed_features:
        print(f'  Dropped: {dropped:<25} (kept: {kept:<25}, r={r:.3f})')

print(f'\n=== Final Selected Price Features ({len(selected_features)}) ===')
for i, f in enumerate(selected_features, 1):
    xgb_imp = combined.loc[f, 'xgb_importance'] if f in combined.index else 0
    tgt_corr = combined.loc[f, 'target_corr'] if f in combined.index else 0
    print(f'  {i:2d}. {f:<28} XGB={xgb_imp:.4f}  |r_target|={tgt_corr:.4f}')

## 11. Quick Validation: Selected Features vs All Features

In [ ]:
results_comparison = []

for ticker in tickers:
    sub = features_clean[features_clean['ticker'] == ticker].sort_values('date')
    y = sub['target_next_close'].values
    split_idx = int(len(y) * 0.8)

    for label, cols in [('All features', feature_cols_all), ('Selected features', selected_features)]:
        X = sub[cols].values
        X_train, X_test = X[:split_idx], X[split_idx:]
        y_train, y_test = y[:split_idx], y[split_idx:]

        model = xgb.XGBRegressor(
            n_estimators=200, max_depth=4, learning_rate=0.05,
            subsample=0.8, colsample_bytree=0.8,
            random_state=42, verbosity=0
        )
        model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
        y_pred = model.predict(X_test)

        rmse = np.sqrt(mean_squared_error(y_test, y_pred))
        mae = mean_absolute_error(y_test, y_pred)
        naive_rmse = np.sqrt(mean_squared_error(y_test[1:], y_test[:-1]))

        results_comparison.append({
            'ticker': ticker,
            'feature_set': label,
            'n_features': len(cols),
            'RMSE': round(rmse, 2),
            'MAE': round(mae, 2),
            'Naive_RMSE': round(naive_rmse, 2),
            'vs_naive': f'{rmse/naive_rmse:.2%}'
        })

results_df = pd.DataFrame(results_comparison)

print('=== All Features vs Selected Features (80/20 time split) ===')
display(results_df.pivot_table(
    index='ticker',
    columns='feature_set',
    values=['RMSE', 'n_features'],
    aggfunc='first'
))

print('\nIf Selected RMSE is similar to or better than All, feature selection is effective.')
print('If worse, we may need to keep more features or adjust the threshold.')

## 12. Save Engineered Features

In [ ]:
output_cols = ['date', 'ticker'] + selected_features + [
    'open', 'high', 'low', 'close', 'volume',
    'target_next_close', 'target_next_return', 'target_direction', 'is_outlier'
]

output_df = features_clean[output_cols].copy()
output_df.to_csv(f'{DATA_DIR}/features_price.csv', index=False)

print(f'Saved: features_price.csv')
print(f'Shape: {output_df.shape}')
print(f'Columns: {list(output_df.columns)}')
print(f'\nDate range: {output_df["date"].min()} to {output_df["date"].max()}')
print(f'Rows per ticker:')
display(output_df.groupby('ticker').size())

## Summary

**Feature engineering pipeline completed:**

1. **Outlier detection** -- Isolation Forest flagging (not removal)
2. **Technical indicators** -- returns, MAs, momentum (RSI, MACD, Stochastic), volatility (BB, ATR, rolling std), price range, volume features, lags, calendar
3. **Cross-ticker features** -- market mean return, relative return, market volatility
4. **Feature importance** -- XGBoost gain + target correlation, dual ranking
5. **Redundancy removal** -- correlation threshold at 0.95 to drop near-duplicate features
6. **Validation** -- selected vs all features comparison against naive baseline

**Next step:** NLP feature engineering (FinBERT sentiment + sentence embeddings) in a separate notebook, then merge with these price features for modeling.